# Saint-Ouen en mutation : recompositions socio-spatiales et dynamiques urbaines depuis les années 2010

*Projet Python pour la data science — Borhane Yahiaoui et Pauline Aurignac - 2026*

## Introduction

Commune limitrophe du nord de Paris en Seine-Saint-Denis, Saint-Ouen-sur-Seine a connu au cours des quinze dernières années une profonde transformation socio-économique. Longtemps associée à un passé industriel marqué par une forte présence ouvrière et une réputation de ville populaire en difficulté, elle s’impose aujourd’hui comme un pôle d’attractivité majeur de la première couronne parisienne. Portée par d’importants projets d’aménagement urbain, l’implantation de sièges sociaux et l’extension de la ligne 14 du métro, la commune attire désormais une population plus aisée et suscite un fort intérêt immobilier, au point d’incarner aux yeux de beaucoup un symbole de réhabilitation urbaine.
L’objectif de ce travail est d’examiner empiriquement à l’aide de Python plusieurs dimensions clés de l’évolution de Saint-Ouen depuis les années 2010. Nous étudions ainsi conjointement la dynamique démographique, l’évolution de la structure socio-économique, les transformations du parc de logements et du marché immobilier, ainsi que l’hétérogénéité des trajectoires entre les différents quartiers de la commune.

*Problématique : Dans quelle mesure les transformations récentes de Saint-Ouen s’accompagnent-elles d’une recomposition socio-spatiale, et comment celle-ci se différencie-t-elle selon les quartiers ?*

Pour y répondre, cette étude couvre la période allant du début des années 2010 aux données les plus récentes disponibles (notamment via le recensement de l’INSEE et les bases de transactions immobilières DVF). Nous étudions les évolutions à l’échelle de la commune et des quartiers, à l’aide du découpage infra-communal des IRIS (Îlots Regroupés pour l’Information Statistique).
Nous présentons dans un premier temps les sources mobilisées et les choix méthodologiques retenus pour harmoniser ces données dans le temps et dans l’espace. Nous analysons ensuite successivement les évolutions démographiques, socio-économiques et résidentielles de la commune, avant de croiser ces différentes dimensions afin de mettre en lumière les dynamiques communes et les contrastes persistants entre quartiers.

Cette approche se veut résolument descriptive : elle vise à mesurer et caractériser des évolutions ainsi que des corrélations spatiales et temporelles observées dans les données publiques, sans prétendre identifier à elle seule les mécanismes causaux à l’œuvre ni démontrer directement des phénomènes d’éviction résidentielle.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import geopandas as gpd

In [2]:
#On définit la police de caractères pour les graphiques
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "DejaVu Sans", "Arial"]

#On améliore la qualité des graphiques
%config InlineBackend.figure_format = 'retina'

In [3]:
#Chargement du fond de carte de l'Île-de-France
gdf_iris = gpd.read_file("IDF_iris.geojson")
gdf_iris.head()

,typ_iris,insee_com,code_iris,nom_iris,dep,geo_point_2d,id,iris,nom_com,geometry
0,A,95051,950510105,Zone d'Activités,95,"[49.018027, 2.182661]",IRIS____0000000950510105,0105,Beauchamp,"POLYGON ((2.18649 49.01053, 2.17345 49.01405, ..."
1,A,75112,751124811,Quinze-Vingts 11,75,"[48.844104, 2.375312]",IRIS____0000000751124811,4811,Paris 12e Arrondissement,"POLYGON ((2.37114 48.84553, 2.37268 48.84568, ..."
2,D,75115,751156023,Jardin Citroën-Cévennes,75,"[48.840354, 2.275217]",IRIS____0000000751156023,6023,Paris 15e Arrondissement,"POLYGON ((2.2746 48.84352, 2.27488 48.84342, 2..."
3,H,92062,920620101,Défense Nord,92,"[48.8913, 2.236816]",IRIS____0000000920620101,0101,Puteaux,"POLYGON ((2.24187 48.89066, 2.24236 48.8905, 2..."
4,H,92014,920140101,Les Bas Coquarts,92,"[48.787231, 2.313623]",IRIS____0000000920140101,0101,Bourg-la-Reine,"POLYGON ((2.31047 48.78489, 2.3097 48.7849, 2...."


In [4]:
#Récupération du tracé de la Ligne 14 (site IDFM)
ligne_14 = gpd.read_file("/Users/borhane/Documents/2A ENSAE/Projet Python 2A/projet_python_2a_ensae/traces-des-lignes-de-transport-en-commun-idfm-2.geojson")
ligne_14.head()

#On sélectionne la zone d'étude correspondant à la commune de Saint-Ouen (code INSEE 93070)
col_insee = 'insee_com' if 'insee_com' in gdf_iris.columns else 'depcom'
zone_etude = gdf_iris[gdf_iris[col_insee].astype(str) == '93070'] #Je force ici à convertir en str pour éviter les problèmes de type de données (les codes iNSEE sont lus comme des int ce qui pose problème)

# Vérifier et convertir si nécessaire le système de coordonnées de la ligne 14 pour qu'il corresponde à celui de la zone d'étude
if ligne_14.crs != zone_etude.crs:
    ligne_14 = ligne_14.to_crs(zone_etude.crs)

DataSourceError: /Users/borhane/Documents/2A ENSAE/Projet Python 2A/projet_python_2a_ensae/traces-des-lignes-de-transport-en-commun-idfm-2.geojson: No such file or directory

In [5]:
#On récupère les emplacements des stations de métro de la ligne 14 (site IDFM)
gdf_stations = gpd.read_file("/Users/borhane/Documents/2A ENSAE/Projet Python 2A/projet_python_2a_ensae/emplacement-des-gares-idf.geojson")

#On vérifie que le système de coordonnées des stations correspond à celui de la zone d'étude, sinon on le convertit
if gdf_stations.crs != zone_etude.crs:
    gdf_stations = gdf_stations.to_crs(zone_etude.crs)

# Récupérer les limites géographiques (bords min et max) de Saint-Ouen
minx, miny, maxx, maxy = zone_etude.total_bounds #On récupère les limites géographiques de la zone d'étude (Saint-Ouen) pour ajuster le zoom du graphique (longitude et latitude)

marge_x = (maxx - minx) * 0.05
marge_y = (maxy - miny) * 0.05

#On sélectionne uniquement les stations de la 14 à Saint-Ouen
stations_st_ouen = gdf_stations.cx[
    minx - marge_x : maxx + marge_x, 
    miny - marge_y : maxy + marge_y] #.cx fait un slicing spaital qui permet de sélectionner les stations dans la zone d'étude (Saint-Ouen) en utilisant les limites géographiques calculées précédemment

DataSourceError: /Users/borhane/Documents/2A ENSAE/Projet Python 2A/projet_python_2a_ensae/emplacement-des-gares-idf.geojson: No such file or directory

In [ ]:
# Visualisation de la zone d'étude et de la ligne 14, carte de référence qu'on va utiliser pour la suite
fig, ax = plt.subplots(figsize=(10, 8))

zone_etude.plot(
    ax=ax,
    color="#f2f2f2",
    edgecolor="#555555",
    linewidth=0.8,
    label="Quartiers (IRIS)"
)
ligne_14.plot(
    ax=ax,
    color="#662483",
    linewidth=3.5,
    zorder=4,
    label="Ligne 14",
)

stations_st_ouen.plot(
    ax=ax,
    color="white",
    edgecolor="#662483",
    linewidth=2.8,
    markersize=100,
    zorder=3,
    label="Stations de métro"
)

#On ajoute les noms des stations de métro sur la carte
texts = []
for _, row in stations_st_ouen.iterrows():
    pt = row.geometry
    t = ax.text(
        pt.x,
        pt.y,
        row["nom_zda"],
        fontsize=10.5,
        fontweight="semibold",
        color="#1f2421",
        zorder=6,
        bbox=dict(
            boxstyle="round,pad=0.28",
            facecolor="white",
            edgecolor="#d0d5dd",
            linewidth=0.7,
            alpha=0.92,
        ),
    )
    texts.append(t)

# Ajouter une petite marge (ex: 10% autour) pour que les tracés ne touchent pas les bords du cadre

# Appliquer le zoom sur les axes du graphique
ax.set_xlim(minx - marge_x, maxx + marge_x)
ax.set_ylim(miny - marge_y, maxy + marge_y)

#On n'affiche pas les axes pour un rendu plus esthétique
ax.set_axis_off()

plt.show()

In [ ]:
#1- Chargement des données immobilières
